# LLM-JEPA Symbolic Regression: Colab Guide

This notebook provides a streamlined guide to training and evaluating the LLM-JEPA Symbolic Regression model in Google Colab, including **Google Drive persistence** for datasets and model weights.

## 1. Setup

Clone the repository and install dependencies.

In [ ]:
import os
!git clone https://github.com/udohchuks/GSOC-LM-JEPA_for_Symbolic_Regression.git
%cd GSOC-LM-JEPA_for_Symbolic_Regression

!pip install -r requirements.txt

## 2. Google Drive Mounting & Persistence

Run this cell to mount your Google Drive and redirect the `checkpoints/` and `cache/` folders to persistent directories in your Drive. This ensures your **model weights** and **precomputed datasets** are saved even after the Colab session ends.

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

# 1. Define your Drive paths
drive_base = '/content/drive/MyDrive/SymbolicRegression'
drive_ckpt_path  = os.path.join(drive_base, 'checkpoints')
drive_cache_path = os.path.join(drive_base, 'cache')

os.makedirs(drive_ckpt_path,  exist_ok=True)
os.makedirs(drive_cache_path, exist_ok=True)

# 2. Redirect local folders to Drive via symlinks
def setup_symlink(local_path, drive_path):
    if os.path.exists(local_path) and not os.path.islink(local_path):
        print(f"Backing up local {local_path} folder...")
        shutil.move(local_path, local_path + '_local_backup')
    
    if not os.path.exists(local_path):
        os.symlink(drive_path, local_path)
        print(f"Symlink created: {local_path} -> {drive_path}")

setup_symlink('checkpoints', drive_ckpt_path)
setup_symlink('cache',       drive_cache_path)

print(f"Success! Checkpoints and Datasets will now persist in Google Drive.")

## 3. Dataset Preparation

### AIF Dataset
Download and extract the AI Feynman dataset.

In [ ]:
import os
import tarfile
import urllib.request

data_dir = './data/'
tar_path = './Feynman_with_units.tar.gz'

if not os.path.exists(os.path.join(data_dir, 'Feynman_with_units')):
    print("Extracting dataset...")
    if not os.path.exists(tar_path):
        url = 'https://www.dropbox.com/s/7kgfr00qpokgz8w/Feynman_with_units.tar.gz?dl=1'
        urllib.request.urlretrieve(url, tar_path)
    
    with tarfile.open(tar_path) as tar:
        tar.extractall(data_dir)
    print("Extraction complete.")
else:
    print("Dataset already exists.")

### Synthetic Dataset (1M+ Equations)
The first time you train with `n_synthetic` set to a large number (e.g., 1M), the model will generate the corpus. Thanks to the symbolics implementation and parallel generation, this will take some time but is much faster on Colab's multi-core CPUs. 

Once generated, the file will be saved to your `cache/` folder (on Google Drive) so you never have to generate it again.

## 4. Training

Start training. Model weights and preprocessed datasets will be saved to your Google Drive folder automatically.

In [ ]:
# Start training with 1M synthetic equations scale
!python -m training.train --config configs/base_config.yaml

## 5. Evaluation & Inference

Since your weights are saved to Google Drive, you can use the **Inference Notebook** (`inference_evaluation.ipynb`) to load them and run tests later.